# LogiScan Stage 3 — Final Training
## 19-Class Logical Fallacy Classifier

**Training data:** 6,712 samples from CoCoLoFa + CoT Logic + Navy0067 + KingTechnician + MrOvkill
**Model:** RoBERTa-base
**Output:** `stage3_production_fallacy.zip`
**Estimated time:** 25-35 minutes on T4x2 GPU (Kaggle)

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets torch scikit-learn tqdm

In [ ]:
# 2. Imports
import json
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}, {torch.cuda.device_count()} device(s) else 'CPU only'}")

### Upload your dataset
Upload `merged_fallacies.json` from your local `data/` folder.

In [ ]:
# 3. Load data from Kaggle datasetDATA_PATH = "/kaggle/input/logiscan-merged-fallacies/merged_fallacies.json"print(f"📁 Loading from: {DATA_PATH}")with open(DATA_PATH) as f:    data = json.load(f)

In [ ]:
# 4. Stratified train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.30, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# 5. Dataset with class-balanced sampling
class FallacyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Tokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Class-balanced sampler for training
class_counts = Counter(y_train)
class_weights = {c: 1.0 / max(count, 1) for c, count in class_counts.items()}
sample_weights = [class_weights[l] for l in y_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = FallacyDataset(X_train, y_train, tokenizer)
val_ds = FallacyDataset(X_val, y_val, tokenizer)
test_ds = FallacyDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

print(f"Batches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

In [ ]:
# 6. Majority baseline
majority_class = Counter(y_test).most_common(1)[0][0]
majority_preds = [majority_class] * len(y_test)
majority_f1 = f1_score(y_test, majority_preds, average="macro")
print(f"Majority baseline (macro F1): {majority_f1:.4f}")
print("Your model must beat this to be useful.")

In [ ]:
# 7. Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
).to(device)

# Focal Loss for class imbalance
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()

criterion = FocalLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epochs = 4
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

params = sum(p.numel() for p in model.parameters())
print(f"Model: {params:,} parameters")
print(f"Epochs: {epochs} | Steps: {total_steps}")

In [ ]:
# 8. Training loop
best_f1 = 0
history = {"train_loss": [], "val_f1": []}

for epoch in range(epochs):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask=mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.3f}"})

    # Validation
    model.eval()
    val_preds, val_truths = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = torch.argmax(out.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_truths.extend(batch["label"].numpy())

    val_f1 = f1_score(val_truths, val_preds, average="macro")
    val_acc = (np.array(val_preds) == np.array(val_truths)).mean()

    history["train_loss"].append(total_loss / len(train_loader))
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, f1_macro={val_f1:.4f}, acc={val_acc:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        Path("stage3_production_fallacy").mkdir(exist_ok=True)
        model.save_pretrained("stage3_production_fallacy")
        tokenizer.save_pretrained("stage3_production_fallacy")
        print(f"  ✅ Saved (f1={val_f1:.4f})")

print(f"\n🏆 Best validation F1: {best_f1:.4f}")
print(f"   Majority baseline: {majority_f1:.4f}")
print(f"   Improvement: {best_f1 - majority_f1:.4f}")

In [ ]:
# 9. Final test evaluation
print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)

model.eval()
test_preds, test_truths = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        preds = torch.argmax(out.logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_truths.extend(batch["label"].numpy())

target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(test_truths, test_preds, target_names=target_names, zero_division=0))

macro_f1 = f1_score(test_truths, test_preds, average="macro")
micro_f1 = f1_score(test_truths, test_preds, average="micro")
print("\n📊 Final Results:")
print(f"   Macro F1: {macro_f1:.4f}")
print(f"   Micro F1: {micro_f1:.4f}")
print(f"   Majority baseline: {majority_f1:.4f}")
print(f"   Improvement: {macro_f1 - majority_f1:.4f}")

In [ ]:
# 10. Per-class breakdown for thin classes
print("\n" + "="*60)
print("PER-CLASS F1 (flag thin classes for your report)")
print("="*60)

per_class_f1 = f1_score(test_truths, test_preds, average=None, zero_division=0)
class_counts_test = Counter(test_truths)

for i, (label, f1_val) in enumerate(zip(target_names, per_class_f1)):
    count = class_counts_test.get(i, 0)
    flag = " ⚠️ THIN" if count < 15 else ""
    bar = "█" * int(f1_val * 20)
    print(f"  {label:30s} F1={f1_val:.3f}  n={count:3d}{flag}  {bar}")

In [ ]:
# 11. Test on real examples
test_examples = [
    ("You cannot trust his argument because he is not a scientist.", "ad_hominem"),
    ("If it rains, the ground is wet. The ground is wet, therefore it rained.", "affirming_consequent"),
    ("So you're saying we should just let the economy collapse?", "straw_man"),
    ("Think of the children who will suffer if this passes.", "appeal_to_emotion"),
    ("Either you support this or you hate our country.", "false_dilemma"),
    ("If we allow this, next thing you know everything falls apart.", "slippery_slope"),
    ("Everyone is buying this product, so it must be good.", "bandwagon"),
    ("After the new policy, crime increased. The policy caused it.", "false_cause"),
]

print("\n" + "="*60)
print("REAL EXAMPLE PREDICTIONS")
print("="*60)

model.eval()
for text, expected in test_examples:
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        top3_idx = torch.topk(probs, 3).indices.cpu().numpy()
        top3_labels = [id2label[i] for i in top3_idx]
        top3_scores = [f"{probs[i].item():.2f}" for i in top3_idx]

    match = "✅" if expected in top3_labels else "❌"
    print(f"\n{match} Expected: {expected}")
    print(f"   Text: {text[:100]}...")
    print(f"   Top-3: {list(zip(top3_labels, top3_scores))}")

In [ ]:
# 10. Save output to Kaggle working directory!zip -r stage3_production_fallacy.zip stage3_production_fallacy/print(f"\n✅ Model saved to /kaggle/working/stage3_production_fallacy.zip")print("\nTo download: commit the notebook, then download from the output tab.")print("\nOn your machine:")print("  unzip stage3_production_fallacy.zip -d models/")